# ViralScope AI - Complete Pipeline

This notebook runs the FULL pipeline:
1. Install dependencies
2. Clone repo
3. Download data from Kaggle
4. Download thumbnails from YouTube CDN
5. Preprocess data (clean, LOO, labels)
6. Tokenize titles with DistilBERT
7. Create train/val/test splits
8. Train model (CNN + NLP + Fusion)
9. Evaluate on test set
10. Download trained model

**Runtime:** ~60-90 minutes with GPU

In [ ]:
#@title 1. Install Dependencies
!pip install kaggle pandas pillow requests tqdm transformers torch torchvision scikit-learn captum gradio matplotlib -q

In [ ]:
#@title 2. Clone Repository & Setup

import os
import sys

if not os.path.exists('/content/ViralScope-AI'):
    !git clone https://github.com/yassine-cmd/ViralScope-AI.git /content/ViralScope-AI

sys.path.insert(0, '/content/ViralScope-AI')
os.chdir('/content/ViralScope-AI')
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title 3. Setup

# CSV files should already be in data/raw/ from previous cell
# If not, upload them when prompted
print("Ready to load CSV files from data/raw/")

In [ ]:
#@title 4. Setup Directories

import yaml

with open('config.yaml', 'r') as f:
    CONFIG = yaml.safe_load(f)

dirs = [
    CONFIG['data']['raw_dir'],
    f"{CONFIG['data']['raw_dir']}/thumbnails",
    CONFIG['data']['processed_dir'],
    CONFIG['data']['tensor_dir'],
    CONFIG['data']['splits_dir'],
    CONFIG['paths']['checkpoints'],
    CONFIG['paths']['results']
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print("Directories created")

# ============================================================
# Validate config has all required keys
# ============================================================
required_config = {
    'training': ['lr_head', 'lr_backbone_cv', 'lr_backbone_nlp', 'weight_decay', 'epochs', 
                  'batch_size', 'num_workers', 'early_stopping_patience', 'scheduler_factor',
                  'scheduler_patience', 'gradient_clip_norm', 'loss_function', 'two_stage'],
    'model': ['cv', 'nlp', 'fusion'],
    'model.cv': ['pretrained', 'feature_dim', 'freeze_backbone'],
    'model.nlp': ['checkpoint', 'max_seq_length', 'feature_dim', 'freeze_backbone'],
    'model.fusion': ['hidden_layers', 'dropout', 'activation'],
    'data': ['raw_dir', 'processed_dir', 'tensor_dir', 'splits_dir'],
    'paths': ['checkpoints', 'best_model', 'training_log']
}

missing = []
for section, keys in required_config.items():
    if '.' in section:
        top, sub = section.split('.')
        if top not in CONFIG:
            missing.append("  CONFIG['{0}'] is missing".format(top))
            continue
        if sub not in CONFIG[top]:
            missing.append("  CONFIG['{0}']['{1}'] is missing".format(top, sub))
    else:
        if section not in CONFIG:
            missing.append("  CONFIG['{0}'] is missing".format(section))
            continue
        for key in keys:
            if key not in CONFIG[section]:
                missing.append("  CONFIG['{0}']['{1}'] is missing".format(section, key))

if missing:
    msg = "Config validation failed - missing required keys:\n" + "\n".join(missing)
    raise KeyError(msg)
print("Config validation passed")
print("=" * 60)


In [ ]:
#@title 5. Upload CSV Files from Local Machine

import os
from google.colab import files

csv_files = [f for f in os.listdir(CONFIG['data']['raw_dir']) if f.endswith('.csv')]

if not csv_files:
    print("No CSV files found. Please upload your YouTube CSV files.")
    print("Upload: USvideos.csv, GBvideos.csv, CAvideos.csv (English-speaking markets only)")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        save_path = os.path.join(CONFIG['data']['raw_dir'], filename)
        with open(save_path, 'wb') as f:
            f.write(content)
        print(f"Saved: {filename}")
    csv_files = [f for f in os.listdir(CONFIG['data']['raw_dir']) if f.endswith('.csv')]

if not csv_files:
    raise ValueError("No CSV files uploaded!")

print(f"âœ“ Found {len(csv_files)} CSV files: {csv_files}")

In [ ]:
#@title 6. Load & Combine CSV Files

import pandas as pd
import numpy as np
import os
import csv

# Load all CSV files and combine
raw_dir = CONFIG['data']['raw_dir']
csv_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv') and f != 'trending.csv']

print(f"Loading {len(csv_files)} CSV files...")

all_dfs = []
for csv_file in csv_files:
    filepath = os.path.join(raw_dir, csv_file)
    try:
        # Default quoting handles titles with commas properly
        df = pd.read_csv(filepath, on_bad_lines='skip', low_memory=False, encoding='utf-8')
        
        # Extract country code from filename
        country = csv_file.replace('videos.csv', '').replace('videos', '')
        
        if 'video_id' not in df.columns:
            if 'id' in df.columns:
                df['video_id'] = df['id']
        
        print(f"  {csv_file}: {len(df)} rows loaded.")
        all_dfs.append(df)
    except Exception as e:
        print(f"  Error loading {csv_file}: {e}")

if not all_dfs:
    raise ValueError("No data loaded from CSV files! Check if files are corrupted or empty.")

# Combine all dataframes
trending_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal combined: {len(trending_df)} rows")

# Validate: only English-speaking markets
english_markets = ['US', 'GB', 'CA']
detected_countries = [f.replace('videos.csv', '').replace('videos', '') for f in csv_files]
non_english = [c for c in detected_countries if c not in english_markets]

if non_english:
    print(f"\nWARNING: Non-English market files detected: {non_english}")
    print("DistilBERT is English-only. Please use only USvideos.csv, GBvideos.csv, CAvideos.csv")

# Save combined data
trending_df.to_csv(f"{raw_dir}/trending.csv", index=False)
print(f"âœ“ Saved: trending.csv")

#@title 7. Clean & Preprocess Data

def clean_dataset(config, raw_dir, processed_dir):
    """Deduplicate and filter data"""
    df = pd.read_csv(f"{raw_dir}/trending.csv", low_memory=False)
    print(f"Raw: {len(df)} rows")

    # Drop NaN video_ids
    if 'video_id' in df.columns:
        df = df.dropna(subset=['video_id'])
        df = df[df['video_id'].astype(str).str.strip() != '']

    # Drop invalid views
    if 'views' in df.columns:
        df['views'] = pd.to_numeric(df['views'], errors='coerce')
        df = df.dropna(subset=['views'])
        df = df[df['views'] > 0]

    # Ensure channel_id
    if 'channel_id' not in df.columns:
        df['channel_id'] = df['channel_title'].astype(str) if 'channel_title' in df.columns else 'unknown'

    # Clean titles
    if 'title' not in df.columns:
        df['title'] = 'Untitled'
    else:
        df['title'] = df['title'].fillna('Untitled').astype(str)
        df = df[df['title'].str.strip() != '']

    # Deduplicate
    if 'video_id' in df.columns:
        df = df.drop_duplicates(subset=['video_id'], keep='last')

    cols = ['video_id', 'title', 'views', 'channel_title', 'channel_id']
    df = df[[c for c in cols if c in df.columns]]
    print(f"Cleaned: {len(df)} rows")
    df.to_csv(f"{processed_dir}/clean_dataset.csv", index=False)
    return df

clean_df = clean_dataset(CONFIG, CONFIG['data']['raw_dir'], CONFIG['data']['processed_dir'])
print(f"âœ“ Saved: clean_dataset.csv")

In [ ]:
#@title 8. Compute LOO Channel Statistics & Labels

def compute_labels(config, df):
    """Compute viral labels using Leave-One-Out"""
    print("Computing viral labels...")
    
    # Group by channel
    channel_stats = df.groupby('channel_id').agg({
        'views': ['sum', 'count', 'mean']
    }).reset_index()
    channel_stats.columns = ['channel_id', 'channel_sum', 'video_count', 'channel_avg']
    
    # Merge back
    df = df.merge(channel_stats, on='channel_id', how='left')
    
    # LOO average
    df['channel_loo_avg_views'] = (df['channel_sum'] - df['views']) / (df['video_count'] - 1).clip(lower=1)
    
    # Viral multiplier
    df['multiplier'] = df['views'] / (df['channel_loo_avg_views'] + 1e-5)
    
    # Filter reliable channels (>= 2 videos)
    reliable = channel_stats[channel_stats['video_count'] >= 2]['channel_id'].tolist()
    df = df[df['channel_id'].isin(reliable)]
    
    # Label
    threshold = config['data']['target_threshold']
    df['is_viral'] = (df['multiplier'] > threshold).astype(int)
    
    # Handle imbalance
    minority_pct = min(df['is_viral'].mean(), 1 - df['is_viral'].mean())
    if minority_pct < 0.10:
        threshold = df['multiplier'].quantile(0.75)
        df['is_viral'] = (df['multiplier'] > threshold).astype(int)
        print(f"Dynamic threshold: {threshold:.2f}")
    
    print(f"Viral: {df['is_viral'].sum()}, Non-viral: {(df['is_viral']==0).sum()}")
    
    cols = ['video_id', 'title', 'views', 'channel_loo_avg_views', 'multiplier', 'is_viral', 'channel_id']
    df = df[cols]
    df.to_csv(f"{config['data']['processed_dir']}/labeled_dataset.csv", index=False)
    print(f"âœ“ Saved: labeled_dataset.csv ({len(df)} rows)")
    return df

labeled_df = compute_labels(CONFIG, clean_df)

In [ ]:
#@title 9. Download Thumbnails

import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

def download_thumbnails(config, labeled_df):
    """Download thumbnails from YouTube CDN"""
    thumb_dir = f"{config['data']['raw_dir']}/thumbnails"
    os.makedirs(thumb_dir, exist_ok=True)
    
    template = config['data']['thumbnail_url_template']
    fallback = config['data']['thumbnail_fallback_url']
    rate_limit = config['data']['thumbnail_rate_limit']
    
    video_ids = labeled_df['video_id'].tolist()
    success, failed = 0, 0
    
    def download_one(vid):
        save_path = os.path.join(thumb_dir, f"{vid}.jpg")
        if os.path.exists(save_path):
            return True
        
        try:
            url = template.format(video_id=vid)
            r = requests.get(url, timeout=5)
            if r.status_code != 200:
                url = fallback.format(video_id=vid)
                r = requests.get(url, timeout=5)
            if r.status_code == 200:
                img = Image.open(BytesIO(r.content))
                if img.size[0] < 120 or img.size[1] < 80:
                    return False
                img.save(save_path)
                return True
        except Exception:
            return False  # Track: download failed
    
    print(f"Downloading thumbnails...")
    for vid in tqdm(video_ids, desc="Thumbnails"):
        if download_one(vid):
            success += 1
        else:
            failed += 1
    
    print(f"âœ“ Downloaded: {success}, Failed: {failed}")
    return success, failed

download_thumbnails(CONFIG, labeled_df)

In [ ]:
#@title 10. Tokenize Titles with DistilBERT

from transformers import DistilBertTokenizer
import torch

def tokenize_titles(config, df):
    """Tokenize titles with DistilBERT"""
    tensor_dir = config['data']['tensor_dir']
    max_len = config['model']['nlp']['max_seq_length']
    checkpoint = config['model']['nlp']['checkpoint']
    
    df['title'] = df['title'].fillna('Untitled').astype(str)
    titles = df['title'].tolist()
    
    tokenizer = DistilBertTokenizer.from_pretrained(checkpoint)
    print(f"Tokenizing {len(titles)} titles...")
    
    encoded = tokenizer(
        titles,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoded['input_ids']
    attention_masks = encoded['attention_mask']
    
    torch.save(input_ids, f"{tensor_dir}/input_ids.pt")
    torch.save(attention_masks, f"{tensor_dir}/attention_masks.pt")
    print(f"âœ“ Saved tensors: {input_ids.shape}")
    return input_ids, attention_masks

tokenize_titles(CONFIG, labeled_df)

In [ ]:
#@title 10. Create Train/Val/Test Splits

import torch

def create_splits(config, df):
    """Create grouped splits by channel_id"""
    splits_dir = config['data']['splits_dir']
    
    from sklearn.model_selection import GroupShuffleSplit
    
    df = df.reset_index(drop=True)
    channels = df['channel_id'].unique()
    np.random.seed(42)
    np.random.shuffle(channels)
    
    n_val = int(0.15 * len(channels))
    n_test = int(0.15 * len(channels))
    
    train_ch = set(channels[n_val+n_test:])
    val_ch = set(channels[:n_val])
    test_ch = set(channels[n_val:n_val+n_test])
    
    train_idx = df[df['channel_id'].isin(train_ch)].index.tolist()
    val_idx = df[df['channel_id'].isin(val_ch)].index.tolist()
    test_idx = df[df['channel_id'].isin(test_ch)].index.tolist()
    
    torch.save(torch.tensor(train_idx), f"{splits_dir}/train_indices.pt")
    torch.save(torch.tensor(val_idx), f"{splits_dir}/val_indices.pt")
    torch.save(torch.tensor(test_idx), f"{splits_dir}/test_indices.pt")
    
    print(f"Splits: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = create_splits(CONFIG, labeled_df)

In [ ]:
#@title 11. Model Architecture

import torch
import torch.nn as nn
import torchvision.models as models
from transformers import DistilBertModel

class CVExtractor(nn.Module):
    def __init__(self, pretrained=True, feature_dim=1280, freeze=True):
        super().__init__()
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(backbone.features, nn.AdaptiveAvgPool2d(1), nn.Flatten())
        if freeze:
            for p in self.features.parameters():
                p.requires_grad = False
    
    def forward(self, x):
        return self.features(x)

class NLPExtractor(nn.Module):
    def __init__(self, checkpoint='distilbert-base-uncased', feature_dim=768, freeze=True):
        super().__init__()
        self.backbone = DistilBertModel.from_pretrained(checkpoint)
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]

class FusionMLP(nn.Module):
    def __init__(self, cv_dim=1280, nlp_dim=768, hidden_layers=[512, 128], dropout=0.4, activation='ReLU'):
        super().__init__()
        layers = []
        prev_dim = cv_dim + nlp_dim
        for hidden in hidden_layers:
            layers.extend([nn.Linear(prev_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU() if activation == 'ReLU' else nn.Tanh(), nn.Dropout(dropout)])
            prev_dim = hidden
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, cv_features, nlp_features):
        combined = torch.cat([cv_features, nlp_features], dim=-1)
        return self.net(combined).squeeze(-1)



In [ ]:
#@title 12. Dataset & DataLoaders

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from transformers import DistilBertTokenizer
from PIL import Image

class ViralScopeDataset(Dataset):
    def __init__(self, split_indices, labeled_df, transform, tokenizer):
        self.df = labeled_df.iloc[split_indices].reset_index(drop=True)
        self.transform = transform
        self.tokenizer = tokenizer
        self.thumb_dir = f"{CONFIG['data']['raw_dir']}/thumbnails"
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vid = row['video_id']
        title = str(row.get('title', 'Untitled'))
        
        # Load image from raw thumbnails directory
        img_path = f"{self.thumb_dir}/{vid}.jpg"
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (128, 128, 128))
        
        image = self.transform(image)
        
        # Tokenize
        enc = self.tokenizer(title, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()
        
        label = torch.tensor(row['is_viral'], dtype=torch.float32)
        
        return {'image': image, 'input_ids': input_ids, 'attention_mask': attention_mask, 'label': label}

# Create SEPARATE transforms for train (augmented) vs val/test (clean)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Training transform (with augmentation from config)
aug = CONFIG.get('augmentation', {})
train_transform = transforms.Compose([
    transforms.Resize(tuple(aug.get('resize_size', [224, 224]))),
    transforms.RandomHorizontalFlip(p=aug.get('horizontal_flip_prob', 0.0)),
    transforms.RandomRotation(degrees=aug.get('rotation_range', [-10, 10])[1]),
    transforms.ColorJitter(
        brightness=aug.get('color_jitter_brightness', 0.2),
        contrast=aug.get('color_jitter_contrast', 0.2),
        saturation=aug.get('color_jitter_saturation', 0.1),
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test transform (NO augmentation)
eval_transform = transforms.Compose([
    transforms.Resize(tuple(aug.get('resize_size', [224, 224]))),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load labeled data
labeled_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv")

# Create datasets with SEPARATE transforms
train_ds = ViralScopeDataset(train_idx, labeled_df, train_transform, tokenizer)
val_ds = ViralScopeDataset(val_idx, labeled_df, eval_transform, tokenizer)
test_ds = ViralScopeDataset(test_idx, labeled_df, eval_transform, tokenizer)

# Compute class weights for imbalanced data (WeightedRandomSampler)
train_labels = labeled_df.iloc[train_idx]['is_viral'].values
class_sample_count = np.array([len(np.where(train_labels == t)[0]) for t in np.unique(train_labels)])
weight = 1. / class_sample_count
samples_weight = torch.from_numpy(np.array([weight[int(t)] for t in train_labels])).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)

batch_size = CONFIG['training']['batch_size']
num_workers = CONFIG['training']['num_workers']

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Class distribution - Viral: {class_sample_count[1]}, Non-viral: {class_sample_count[0]}")
print(f"Using WeightedRandomSampler to balance classes")
print(f"Batch size: {batch_size}, Num workers: {num_workers}")

In [ ]:
#@title 13. Focal Loss & Training Functions (Fixed)

import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

# Fixed FocalLoss with proper alpha handling (from models/losses.py)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        
        # FIXED: Properly handle alpha weights
        if self.alpha is not None:
            alpha_t = self.alpha[1] * targets + self.alpha[0] * (1 - targets)
            focal_weight = focal_weight * alpha_t
        
        loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        return loss

def build_criterion(CONFIG, train_labels):
    """Build FocalLoss with auto-computed alpha from class distribution"""
    alpha = None
    
    # Auto-compute alpha from training set class distribution
    if CONFIG['training'].get('focal_loss_alpha') is None:
        n_pos = (train_labels == 1).sum()
        n_neg = (train_labels == 0).sum()
        total = n_pos + n_neg
        alpha_neg = total / (2.0 * n_neg) if n_neg > 0 else 1.0
        alpha_pos = total / (2.0 * n_pos) if n_pos > 0 else 1.0
        alpha = torch.tensor([float(alpha_neg), float(alpha_pos)], dtype=torch.float32)
        print(f"Auto-computed FocalLoss alpha: {alpha.tolist()}")
    
    return FocalLoss(
        alpha=alpha,
        gamma=CONFIG['training'].get('focal_loss_gamma', 2.0),
    )

def compute_metrics(preds, targets, threshold=0.5):
    preds_binary = (preds >= threshold).float()
    tp = ((preds_binary == 1) & (targets == 1)).sum().item()
    tn = ((preds_binary == 0) & (targets == 0)).sum().item()
    fp = ((preds_binary == 1) & (targets == 0)).sum().item()
    fn = ((preds_binary == 0) & (targets == 1)).sum().item()
    acc = (tp + tn) / (tp + tn + fp + fn + 1e-10)
    prec = tp / (tp + fp + 1e-10)
    rec = tp / (tp + fn + 1e-10)
    f1 = 2 * prec * rec / (prec + rec + 1e-10)
    auc = roc_auc_score(targets.numpy(), preds.numpy())
    pr_auc = average_precision_score(targets.numpy(), preds.numpy())
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc_roc': auc, 'pr_auc': pr_auc}

def train_epoch(model, loader, criterion, optimizer, device, clip_norm=1.0):
    """Train one epoch with gradient clipping for stability"""
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(images, input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        # Gradient clipping - critical when backbones are unfrozen
        if clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(logits).detach().cpu())
        all_labels.extend(labels.cpu())
    
    all_preds = torch.stack(all_preds)
    all_labels = torch.stack(all_labels)
    metrics = compute_metrics(all_preds, all_labels)
    metrics['loss'] = total_loss / len(loader)
    return metrics

@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    """Validate one epoch"""
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(images, input_ids, attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(logits).cpu())
        all_labels.extend(labels.cpu())
    
    all_preds = torch.stack(all_preds)
    all_labels = torch.stack(all_labels)
    metrics = compute_metrics(all_preds, all_labels)
    metrics['loss'] = total_loss / len(loader)
    return metrics

def unfreeze_backbone(model, backbone_name, lr, optimizer):
    """Unfreeze a backbone and add its params to the optimizer with a low LR"""
    backbone = getattr(model, backbone_name)
    for param in backbone.parameters():
        param.requires_grad = True
    optimizer.add_param_group({
        'params': [p for p in backbone.parameters() if p.requires_grad],
        'lr': lr,
    })
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f"  Unfroze {backbone_name}: {trainable:,} params at lr={lr}")

print("\u2713 Training functions defined with proper FocalLoss")

In [ ]:
#@title 14. Initialize Model & Optimizer (from config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Update model to use config values for fusion head
class ViralScopeModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.cv_extractor = CVExtractor(
            pretrained=config['model']['cv']['pretrained'],
            feature_dim=config['model']['cv']['feature_dim'],
            freeze=config['model']['cv']['freeze_backbone']
        )
        self.nlp_extractor = NLPExtractor(
            checkpoint=config['model']['nlp']['checkpoint'],
            feature_dim=config['model']['nlp']['feature_dim'],
            freeze=config['model']['nlp']['freeze_backbone']
        )
        self.fusion = FusionMLP(
            cv_dim=config['model']['cv']['feature_dim'],
            nlp_dim=config['model']['nlp']['feature_dim'],
            hidden_layers=config['model']['fusion']['hidden_layers'],
            dropout=config['model']['fusion']['dropout'],
            activation=config['model']['fusion']['activation']
        )
    
    def forward(self, images, input_ids, attention_mask):
        cv_features = self.cv_extractor(images)
        nlp_features = self.nlp_extractor(input_ids, attention_mask)
        return self.fusion(cv_features, nlp_features)
    
    def predict_proba(self, images, input_ids, attention_mask):
        return torch.sigmoid(self.forward(images, input_ids, attention_mask))

model = ViralScopeModel(CONFIG).to(device)

# Only fusion head params initially (backbones frozen for head warmup)
head_params = [p for p in model.parameters() if p.requires_grad]

training_cfg = CONFIG['training']

optimizer = optim.AdamW(
    head_params,
    lr=training_cfg['lr_head'],
    weight_decay=training_cfg['weight_decay'],
    betas=tuple(training_cfg.get('betas', [0.9, 0.999]))
)

# Build criterion with auto-computed alpha
train_labels_tensor = torch.from_numpy(train_labels.astype(np.float32))
criterion = build_criterion(CONFIG, train_labels_tensor)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    factor=training_cfg['scheduler_factor'], 
    patience=training_cfg['scheduler_patience']
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in head_params)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable params:     {trainable_params:,} (head only)")
print(f"Learning rate (head): {training_cfg['lr_head']}")
print(f"\nFusion hidden layers: {CONFIG['model']['fusion']['hidden_layers']}")
print(f"Fusion dropout:       {CONFIG['model']['fusion']['dropout']}")

In [ ]:
#@title 15. Two-Stage Training Loop

epochs = CONFIG['training']['epochs']
clip_norm = CONFIG['training'].get('gradient_clip_norm', 1.0)
two_stage = CONFIG['training'].get('two_stage', {})

head_warmup_epochs = two_stage.get('head_warmup_epochs', 5) if two_stage.get('enabled') else epochs
best_metric = 0.0
best_state = None
patience_counter = 0
patience = CONFIG['training']['early_stopping_patience']
training_log = []
backbones_unfrozen = False

print(f"\n{'='*80}")
print(f"Starting {epochs} epochs of two-stage training...")
print(f"  Stage 1: Head warmup for {head_warmup_epochs} epochs (backbones frozen)")
if two_stage.get('enabled'):
    print(f"  Stage 2: Full fine-tuning (backbones unfrozen at epoch {head_warmup_epochs + 1})")
print(f"{'='*80}\n")

print(f"{'Epoch':>5} | {'Stage':>7} | {'Train Loss':>10} | {'Train Acc':>8} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val F1':>7} | {'Val PR-AUC':>10}")
print("-" * 100)

for epoch in range(epochs):
    
    # Stage 2: Unfreeze backbones
    if two_stage.get('enabled') and epoch == head_warmup_epochs and not backbones_unfrozen:
        print(f"\n{'='*100}")
        print(f"  STAGE 2: Unfreezing backbones at epoch {epoch + 1}")
        print(f"{'='*100}")
        
        if two_stage.get('unfreeze_cv', True):
            unfreeze_backbone(model, 'cv_extractor', CONFIG['training']['lr_backbone_cv'], optimizer)
        if two_stage.get('unfreeze_nlp', True):
            unfreeze_backbone(model, 'nlp_extractor', CONFIG['training']['lr_backbone_nlp'], optimizer)
        
        backbones_unfrozen = True
        trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable params now: {trainable_now:,}")
        
        # Reset early stopping and scheduler for stage 2
        patience_counter = 0
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max',
            factor=CONFIG['training']['scheduler_factor'],
            patience=CONFIG['training']['scheduler_patience']
        )
        print()
    
    stage = "head" if not backbones_unfrozen else "full"
    
    # Train with gradient clipping
    train_m = train_epoch(model, train_loader, criterion, optimizer, device, clip_norm)
    val_m = validate_epoch(model, val_loader, criterion, device)
    
    scheduler.step(val_m['pr_auc'])
    
    log = {
        'epoch': epoch + 1,
        'stage': stage,
        'train_loss': train_m['loss'],
        'train_accuracy': train_m['accuracy'],
        'train_f1': train_m['f1'],
        'val_loss': val_m['loss'],
        'val_accuracy': val_m['accuracy'],
        'val_f1': val_m['f1'],
        'val_auc_roc': val_m['auc_roc'],
        'val_pr_auc': val_m['pr_auc']
    }
    training_log.append(log)
    
    print(
        f"{epoch+1:>5} | {stage:>7} | {train_m['loss']:>10.4f} | {train_m['accuracy']:>8.4f} | "
        f"{val_m['loss']:>10.4f} | {val_m['accuracy']:>8.4f} | {val_m['f1']:>7.4f} | {val_m['pr_auc']:>10.4f}"
    )
    
    if val_m['pr_auc'] > best_metric:
        best_metric = val_m['pr_auc']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print(f"  -> Saved best model (PR-AUC: {best_metric:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs")
            break

print(f"\nTraining complete. Best PR-AUC: {best_metric:.4f}")

In [ ]:
#@title 16. Evaluate on Test Set (Threshold from Validation)

from sklearn.metrics import confusion_matrix, classification_report

if best_state is not None:
    model.load_state_dict(best_state)
    model.eval()

@torch.no_grad()
def collect_predictions(model, dataloader, device):
    all_probs, all_labels = [], []
    for batch in dataloader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']
        logits = model(images, input_ids, attention_mask)
        all_probs.extend(torch.sigmoid(logits).cpu())
        all_labels.extend(labels)
    return torch.stack(all_probs).numpy(), torch.stack(all_labels).numpy()

# Step 1: Find optimal threshold on VALIDATION set (no leakage)
print("Finding optimal threshold on validation set...")
val_probs, val_labels = collect_predictions(model, val_loader, device)

precision, recall, thresholds = precision_recall_curve(val_labels, val_probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
val_best_f1 = float(f1_scores[best_idx])

# Sanity check: if threshold is unreasonably low, clamp to 0.3
if optimal_threshold < 0.2:
    print(f"Note: Threshold {optimal_threshold:.4f} is below 0.2 - this can be normal for imbalanced data")

print(f"Optimal threshold: {optimal_threshold:.4f} (Val F1: {val_best_f1:.4f})")

# Step 2: Evaluate on TEST set with the validation-derived threshold
print("\nEvaluating on test set...")
test_probs, test_labels = collect_predictions(model, test_loader, device)

# Report with optimal threshold
test_preds = (test_probs > optimal_threshold).astype(int)
metrics = {
    'accuracy': float((test_preds == test_labels).mean()),
    'f1': float(f1_score(test_labels, test_preds)),
    'auc_roc': float(roc_auc_score(test_labels, test_probs)),
    'pr_auc': float(average_precision_score(test_labels, test_probs)),
    'confusion_matrix': confusion_matrix(test_labels, test_preds).tolist()
}

# Also report with fixed 0.5 threshold for comparison
test_preds_fixed = (test_probs > 0.5).astype(int)
metrics_fixed = {
    'accuracy': float((test_preds_fixed == test_labels).mean()),
    'f1': float(f1_score(test_labels, test_preds_fixed)),
    'confusion_matrix': confusion_matrix(test_labels, test_preds_fixed).tolist()
}

print("="*60)
print("TEST SET EVALUATION (Threshold from Validation)")
print("="*60)
print(f"\n--- With optimal threshold ({optimal_threshold:.4f}) ---")
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"AUC-ROC:   {metrics['auc_roc']:.4f}")
print(f"PR-AUC:    {metrics['pr_auc']:.4f}")
print(f"\nConfusion Matrix:\n{metrics['confusion_matrix']}")

print(f"\n--- With fixed threshold (0.50) ---")
print(f"Accuracy:  {metrics_fixed['accuracy']:.4f}")
print(f"F1-Score:  {metrics_fixed['f1']:.4f}")
print(f"Confusion Matrix:\n{metrics_fixed['confusion_matrix']}")
print("="*60)

In [ ]:
#@title 17. Save Model

import os
import pandas as pd

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

if best_state is not None:
    torch.save(best_state, 'models/best_model.pt')
    print("âœ“ Saved: models/best_model.pt")

pd.DataFrame(training_log).to_csv('results/training_log.csv', index=False)
print("âœ“ Saved: results/training_log.csv")

In [ ]:
#@title 18. Download from Colab

from google.colab import files

# Download model
if os.path.exists('models/best_model.pt'):
    files.download('models/best_model.pt')
    print("Downloaded: best_model.pt")

# Download training log
if os.path.exists('results/training_log.csv'):
    files.download('results/training_log.csv')
    print("Downloaded: training_log.csv")